# Limpeza de Dados — Cyclistic Bike-Share

Este notebook documenta o processo completo de verificação de integridade e
limpeza dos dados históricos de viagens da Cyclistic (abr. 2025 - mar. 2026).
Cada decisão tomada é registrada com a respectiva justificativa, de forma que
o processo seja rastreável e reproduzível.

**Ferramenta:** Python + DuckDB  
**Fonte dos dados:** Divvy/Lyft — disponibilizados publicamente via Motivate International Inc.  
**Licença:** https://divvybikes.com/data-license-agreement


In [1]:
#Preparando o ambiente de limpeza
import duckdb
import pandas as pd
import os

os.makedirs('../database', exist_ok=True)
con = duckdb.connect('../database/cyclistic_project.duckdb')

print('Conexão aberta.')

Conexão aberta.


In [2]:
#Criando uma View para referenciar os arquivos CSV direto da origem
con.execute("""
    CREATE OR REPLACE VIEW arquivos_csv AS
    SELECT *
    FROM read_csv_auto('../data/*.csv')
""");

## 2. Visão Geral dos Dados

Antes de qualquer verificação ou limpeza, precisamos entender com o que estamos
trabalhando: quantos registros existem, quais colunas compõem o dataset, quais
são os tipos de dados e como os registros estão distribuídos.  
Essa visão inicial orienta todas as decisões que virão a seguir.



In [3]:
#Quantidade total de registros
con.execute("SELECT COUNT(*) AS total_registros FROM arquivos_csv").df()

,total_registros
0,5620544


In [4]:
#Checando a estrutura do nosso dataset
con.execute("DESCRIBE SELECT * FROM arquivos_csv").df()

,column_name,column_type,null,key,default,extra
0,ride_id,VARCHAR,YES,None,None,None
1,rideable_type,VARCHAR,YES,None,None,None
2,started_at,TIMESTAMP,YES,None,None,None
3,ended_at,TIMESTAMP,YES,None,None,None
4,start_station_name,VARCHAR,YES,None,None,None
5,start_station_id,VARCHAR,YES,None,None,None
6,end_station_name,VARCHAR,YES,None,None,None
7,end_station_id,VARCHAR,YES,None,None,None
8,start_lat,DOUBLE,YES,None,None,None
9,start_lng,DOUBLE,YES,None,None,None


## 3. Verificações de Integridade

Antes de modificar qualquer registro, verificamos a qualidade dos dados.  
As verificações seguem as dimensões de qualidade definidas pelo framework DAMA-DMBOK
(Data Management Body of Knowledge), padrão amplamente adotado pelas empresas.

Das 6 dimensões do framework, 4 são verificáveis neste dataset:  
- **3.1 Unicidade (Uniqueness):**   registros duplicados  
- **3.2 Completude (Completeness):** dados nulos  
- **3.3 Consistência (Consistency):**  dados que se contradizem entre si  
- **3.4 Validade (Validity):**     dados com valores fora do permitido  

As outras duas dimensões não são aplicáveis aqui:  
- **Acurácia (Accuracy):** verificar se os dados refletem a realidade - foge do nosso alcance para essa análise.  
- **Atualidade (Timeliness):** dados atuais - os dados foram coletados com um intervalo especifico de tempo.


### 3.1 Uniqueness (Unicidade)

Cada viagem deveria ter um identificador único (`ride_id`).  
Se houver duplicatas nessa coluna, os registros em questão representariam a mesma viagem
inserida duas vezes, distorcendo qualquer contagem ou agregação feita na análise.  
Além disso, verificamos se há linhas completamente idênticas em todas as colunas
(exceto `ride_id`), o que indicaria um mesmo evento registrado mais de uma vez, mesmo que com identificador único diferente.


In [5]:
#Verificando se tem duplicatas em ride_id
con.execute("""
    SELECT COUNT(*) AS ride_id_duplicados
    FROM (
        SELECT ride_id
        FROM arquivos_csv
        GROUP BY ride_id
        HAVING COUNT(*) > 1
    ) t
""").df()

,ride_id_duplicados
0,0


In [6]:
#Verificando se tem duplicatas em todo o resto, menos ride_id
con.execute("""
    SELECT COUNT(*) AS registros_duplicados
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY rideable_type, started_at, ended_at,
                                start_station_name, start_station_id,
                                end_station_name, end_station_id,
                                start_lat, start_lng, end_lat, end_lng,
                                member_casual
                   ORDER BY rideable_type
               ) AS rn
        FROM arquivos_csv
    ) t
    WHERE rn > 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,registros_duplicados
0,0


Ambas as verificações retornaram zero ocorrências: não há duplicatas
em `ride_id` nem linhas completamente idênticas no dataset.  
A unicidade dos registros está preservada — nenhuma ação necessária.

### 3.2 Completeness (Completude)

Verificamos a presença de valores ausentes (nulos) no dataset.  
O objetivo é identificar onde existem nulos, em que volume,
e se seguem algum padrão — o que indicaria uma causa sistemática.


In [7]:
#Verificando nulos por coluna
total_registros_df = con.execute("SELECT COUNT(*) FROM arquivos_csv").df()
total_registros = total_registros_df.iloc[0, 0]

con.execute(f"""
    SELECT
        ROUND(((COUNT(*) - COUNT(ride_id)) * 100.0) / {total_registros}, 2) AS pct_ride_id,
        ROUND(((COUNT(*) - COUNT(rideable_type)) * 100.0) / {total_registros}, 2) AS pct_rideable_type,
        ROUND(((COUNT(*) - COUNT(started_at)) * 100.0) / {total_registros}, 2) AS pct_started_at,
        ROUND(((COUNT(*) - COUNT(ended_at)) * 100.0) / {total_registros}, 2) AS pct_ended_at,
        ROUND(((COUNT(*) - COUNT(start_station_name)) * 100.0) / {total_registros}, 2) AS pct_start_station_name,
        ROUND(((COUNT(*) - COUNT(start_station_id)) * 100.0) / {total_registros}, 2) AS pct_start_station_id,
        ROUND(((COUNT(*) - COUNT(end_station_name)) * 100.0) / {total_registros}, 2) AS pct_end_station_name,
        ROUND(((COUNT(*) - COUNT(end_station_id)) * 100.0) / {total_registros}, 2) AS pct_end_station_id,
        ROUND(((COUNT(*) - COUNT(end_lat)) * 100.0) / {total_registros}, 2) AS pct_end_lat,
        ROUND(((COUNT(*) - COUNT(end_lng)) * 100.0) / {total_registros}, 2) AS pct_end_lng
    FROM arquivos_csv
""").df()

,pct_ride_id,pct_rideable_type,pct_started_at,pct_ended_at,pct_start_station_name,pct_start_station_id,pct_end_station_name,pct_end_station_id,pct_end_lat,pct_end_lng
0,0.0,0.0,0.0,0.0,21.26,21.26,22.4,22.4,0.1,0.1


Os nulos estão concentrados, principalmente, nas colunas de estação:
`start_station_name`, `start_station_id`, `end_station_name`, `end_station_id`.
Há também uma porcentagem de 0.1% de registros nulos nas colunas `end_lat` e `end_lng` — pouco relevante.  
Colunas críticas como `ride_id`, `started_at`, `ended_at` e `member_casual`
não apresentam nulos — boa integridade nos campos mais relevantes para a análise.  

Os números de valores nulos em colunas pares (`start_station_name` / `start_station_id` e
`end_station_name` / `end_station_id`) são idênticos, o que sugere que os nulos são sempre compartilhados.
Verificaremos essa relação antes de prosseguir.  

Isso levanta uma questão: por que as colunas de estação têm uma porcentagem tão alta de valores ausentes?  
Antes de qualquer remoção, investigamos se esses nulos seguem algum padrão identificável dentro do próprio dataset.


Primeiro verificamos se as colunas `start_station_name` e `start_station_id` compartilham
seus valores nulos, como os números sugerem.  
O mesmo será feito com `end_station_name` e `end_station_id`.  
Vou fazer isso usando uma abordagem de exclusão mútua = buscar qualquer registro em que um nulo da coluna A não esteja acompanhado de um nulo da coluna B.

In [8]:
#Verificando se start_station_name e start_station_id compartilham nulos
con.execute("""
    SELECT COUNT(*) AS linhas_divergentes
    FROM arquivos_csv
    WHERE (start_station_name IS NULL AND start_station_id IS NOT NULL)
       OR (start_station_name IS NOT NULL AND start_station_id IS NULL)
""").df()

,linhas_divergentes
0,0


In [9]:
#Verificando se end_station_name e end_station_id compartilham nulos
con.execute("""
    SELECT COUNT(*) AS linhas_divergentes
    FROM arquivos_csv
    WHERE (end_station_name IS NULL AND end_station_id IS NOT NULL)
       OR (end_station_name IS NOT NULL AND end_station_id IS NULL)
""").df()

,linhas_divergentes
0,0


Os resultados confirmam que essas colunas compartilham seus valores nulos.  

Checamos então se: sempre que um registro tiver valores nulos em `start_station_name` ele também
terá valores nulos em `end_station_name`, e vice-versa.
Em seguida, verificamos a relação dos nulos em `end_lat` e `end_lng` com os nulos em estação.  

O objetivo é estabelecer um filtro simples que abranja todos os valores nulos do dataset para a sequência da limpeza.


In [10]:
#Verificando a relação dos nulos entre start_station_name e end_station_name
con.execute("""
    SELECT
        COUNT(*) AS "Ambos Nulos",
        (SELECT COUNT(*) FROM arquivos_csv WHERE end_station_name IS NULL OR start_station_name IS NULL) AS "Um deles Nulo"
    FROM arquivos_csv
    WHERE start_station_name IS NULL AND end_station_name IS NULL
""").df()

,Ambos Nulos,Um deles Nulo
0,572867,1881299


In [11]:
#Verificando se há relação entre nulos em end_lat/end_lng e nulos em estação
con.execute("""
    SELECT COUNT(*) AS "Exceções à regra"
    FROM arquivos_csv
    WHERE (end_lat IS NULL OR end_lng IS NULL) 
      AND end_station_name IS NOT NULL
""").df()

,Exceções à regra
0,0


Os resultados mostram que sempre que um valor for nulo em `end_lat`/`end_lng` ele também será nulo
em `end_station_name`. Tudo dentro do esperado.  

Com isso, definimos um filtro que abranja todos os valores nulos da forma mais simplificada.


In [12]:
#Filtro que abrange todos os registros com nulos
FILTRO_NULOS = 'start_station_name IS NULL OR end_station_name IS NULL'

In [13]:
#Testando se o filtro abrange todos os nulos do dataset
con.execute(f"""
    SELECT COUNT(*)
    FROM arquivos_csv
    WHERE {FILTRO_NULOS}
    UNION ALL
    SELECT COUNT(*)
    FROM arquivos_csv
    WHERE (start_station_name IS NULL OR start_station_id IS NULL)
       OR (end_station_name IS NULL OR end_station_id IS NULL)
       OR (end_lat IS NULL OR end_lng IS NULL)
""").df()

,count_star()
0,1881299
1,1881299


Agora vou verificar se os nulos nas estações têm alguma relação com o tipo de bicicleta usado.  
**Hipótese:** Esses nulos têm relação com o uso de bicicletas elétricas, pois as mesmas podem ser
travadas em racks públicos e outras estruturas autorizadas fora das estações da empresa,
tornando o campo de estação "ocasionalmente nulo".  

Para evitar o viés de seleção, vou tentar primeiro refutar a hipótese.
Farei isso relacionando os nulos com o tipo de usuário (membro ou casual).


In [14]:
#Distribuição dos nulos por tipo de usuário
con.execute(f"""
    SELECT COALESCE(member_casual, 'Total') AS "Usuário", COUNT(*) AS Total
    FROM arquivos_csv
    WHERE {FILTRO_NULOS}
    GROUP BY ROLLUP(member_casual)
    ORDER BY Total
""").df()

,Usuário,Total
0,casual,683961
1,member,1197338
2,Total,1881299


Os valores nulos estão distribuídos de forma proporcional à população —
não há nenhuma relação especial entre os nulos e o tipo de usuário (membro ou casual).  

Isto posto, vou testar minha hipótese.


In [15]:
#Checando se há relação entre nulos de estação e tipo de bicicleta

con.execute(f"""
    SELECT rideable_type, COUNT(*) AS total
    FROM arquivos_csv
    WHERE {FILTRO_NULOS}
    GROUP BY 1
    ORDER BY total DESC
""").df()

,rideable_type,total
0,electric_bike,1875431
1,classic_bike,5868


99,7% dos registros com valores nulos pertencem a bicicletas elétricas.  
Esse padrão confirma a hipótese: a ausência de estação é um comportamento esperado
para e-bikes, que não precisam ser devolvidas em estação fixa.  
Registros de bicicletas clássicas com estação nula (0,3%) serão tratados como falhas
de registro e removidos na etapa de limpeza, pois bicicletas clássicas exigem devolução em estação.


### 3.3 Consistency (Consistência)

Verificamos se os dados respeitam regras lógicas internas.  
A principal regra aplicável neste dataset é a ordem cronológica:
o horário de início de uma viagem (`started_at`) deve ser sempre
anterior ao horário de fim (`ended_at`).  
Registros que violam essa regra são logicamente impossíveis e precisam ser descartados.


In [16]:
#Registros com started_at > ended_at
con.execute("""
    SELECT
        (SELECT COUNT(*) FROM arquivos_csv WHERE started_at > ended_at) AS timestamps_impossiveis
""").df()


,timestamps_impossiveis
0,29


Esses registros são *fisicamente* impossíveis e serão removidos durante a etapa de limpeza.


### 3.4 Validity (Validade)

Verificamos se os valores de cada coluna estão dentro do domínio esperado —
seja um conjunto fixo de categorias, um intervalo temporal válido ou
um padrão geográfico conhecido. Um valor pode estar *presente* e ainda assim
ser inválido se estiver fora do domínio definido para aquela coluna.  

Realizamos quatro verificações:  
- **3.4.1** Colunas categóricas (`rideable_type`, `member_casual`)  
- **3.4.2** Intervalo temporal (`started_at`, `ended_at`)  
- **3.4.3** Coordenadas geográficas (`start_lat/lng`, `end_lat/lng`)  
- **3.4.4** Nomes de estações com valores de teste ou sistema  


#### 3.4.1 Colunas Categóricas


In [17]:
#Valores únicos em rideable_type
con.execute("""
    SELECT DISTINCT rideable_type FROM arquivos_csv
""").df()

,rideable_type
0,classic_bike
1,electric_bike


In [18]:
#Valores únicos em member_casual
con.execute("""
    SELECT DISTINCT member_casual FROM arquivos_csv
""").df()

,member_casual
0,member
1,casual


As colunas categóricas apresentam os valores esperados.


#### 3.4.2 Intervalo Temporal

Verificamos se as datas estão dentro do intervalo esperado pelo dataset (Abril de 2025 – Março de 2026).


In [19]:
#Intervalo de datas do dataset
con.execute("""
    SELECT min(started_at), min(ended_at), max(started_at), max(ended_at)
    FROM arquivos_csv
""").df()

,min(started_at),min(ended_at),max(started_at),max(ended_at)
0,2025-03-31 23:17:22.078,2025-04-01 00:02:33.796,2026-03-31 23:55:56.848,2026-03-31 23:59:47.821


O resultado mostra que há registros marginalmente fora do escopo, mas em margem plausível:
viagens iniciadas no dia 31 de março e finalizadas no dia 1º de abril são perfeitamente válidas.


#### 3.4.3 Coordenadas Geográficas

As coordenadas registradas no dataset deveriam estar dentro da área de Chicago.  

Área aproximada de Chicago:  
- Latitude: 41.6° a 42.1°  
- Longitude: −88.0° a −87.5°  

Os valores que estiverem dentro dessa margem, ou muito próximos, serão validados.


In [20]:
#Menores coordenadas registradas
con.execute("""
    SELECT
        min(start_lat) AS "menor latitude inicial",
        min(end_lat)   AS "menor latitude final",
        min(start_lng) AS "menor longitude inicial",
        min(end_lng)   AS "menor longitude final"
    FROM arquivos_csv
""").df()

,menor latitude inicial,menor latitude final,menor longitude inicial,menor longitude final
0,41.648501,41.49,-87.89,-88.1


In [21]:
#Maiores coordenadas registradas
con.execute("""
    SELECT
        max(start_lat) AS "maior latitude inicial",
        max(end_lat)   AS "maior latitude final",
        max(start_lng) AS "maior longitude inicial",
        max(end_lng)   AS "maior longitude final"
    FROM arquivos_csv
""").df()

,maior latitude inicial,maior latitude final,maior longitude inicial,maior longitude final
0,42.07,42.21,-87.52,-87.42


Todos os valores estão dentro do aceitável para nossa análise.


#### 3.4.4 Nomes de Estações com Valores de Teste ou Sistema

Sistemas de bike-share frequentemente contêm registros gerados por testes de engenharia
ou manutenção. Esses registros possuem nomes de estação como `"TEST"`, `"REPAIR"` ou `"CHECKING"`.  

Ao contrário dos nulos, esses valores estão *presentes* mas não representam
viagens reais de usuários — incluí-los na análise distorceria os resultados.


In [22]:
#Verificando a palavra 'TEST'
con.execute("""
    SELECT COUNT(*) AS "registros com TEST"
    FROM arquivos_csv
    WHERE start_station_name LIKE '%TEST%' OR end_station_name LIKE '%TEST%'
""").df()

,registros com TEST
0,0


In [23]:
#Verificando a palavra 'REPAIR'
con.execute("""
    SELECT COUNT(*) AS "registros com REPAIR"
    FROM arquivos_csv
    WHERE start_station_name LIKE '%REPAIR%' OR end_station_name LIKE '%REPAIR%'
""").df()

,registros com REPAIR
0,0


In [24]:
#Verificando a palavra 'CHECKING'
con.execute("""
    SELECT COUNT(*) AS "registros com CHECKING"
    FROM arquivos_csv
    WHERE start_station_name LIKE '%CHECKING%' OR end_station_name LIKE '%CHECKING%'
""").df()

,registros com CHECKING
0,0


Nenhum registro de teste identificado.


## 4. Limpeza dos Dados

Com base nas verificações de integridade, os registros que ferem a qualidade do dado
serão removidos. Cada etapa é acompanhada de uma justificativa e verificada antes
de prosseguir.  
O resultado final será a tabela `cyclistic`, contendo apenas os registros válidos
e com as colunas derivadas necessárias para a análise.


### 4.1 Remoção de Registros Logicamente Inválidos

Registros com `started_at > ended_at` são fisicamente impossíveis e são descartados.  
Registros de bicicletas clássicas sem informação de estação também são descartados:
bicicletas clássicas exigem devolução em estação — a ausência indica falha de registro,
não um comportamento operacional esperado (ao contrário das elétricas).


In [25]:
#Removendo registros logicamente inválidos
con.execute("""
    CREATE OR REPLACE TABLE etapa_01 AS
    SELECT *
    FROM arquivos_csv
    WHERE NOT (
        started_at > ended_at
        OR (
            (start_station_name IS NULL OR end_station_name IS NULL)
            AND rideable_type = 'classic_bike'
        )
    )
""")

con.execute("""
    SELECT
        (SELECT COUNT(*) FROM arquivos_csv) AS registros_totais,
        (SELECT COUNT(*) FROM etapa_01) AS registros_etapa_01,
        (SELECT COUNT(*) FROM arquivos_csv) - (SELECT COUNT(*) FROM etapa_01) AS removidos_etapa_01
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,registros_totais,registros_etapa_01,removidos_etapa_01
0,5620544,5614647,5897


A coluna `removidos_etapa_01` mostra o total de registros excluídos:
timestamps impossíveis e bicicletas clássicas sem informação de estação.

### 4.2 Adição de Colunas Derivadas

Calculamos `ride_duration_min` (duração da viagem em minutos) e `day_of_week`
(dia da semana) a partir de `started_at` e `ended_at`.  
Essas colunas são necessárias para a análise de padrões de uso entre membros
e usuários casuais.

In [27]:
#Criando colunas derivadas
con.execute("""
    CREATE OR REPLACE TABLE etapa_02 AS
    SELECT
        *,
        ROUND(EPOCH(ended_at - started_at) / 60, 2) AS ride_duration_min,
        DAYOFWEEK(started_at) AS day_of_week
    FROM etapa_01
""")

con.execute("SELECT * FROM etapa_02 LIMIT 10").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_min,day_of_week
0,AF3863596DF9D94B,classic_bike,2025-04-27 14:29:34.619,2025-04-27 14:36:23.584,Troy St & Elston Ave,15631,Richmond St & Diversey Ave,15645,41.945244,-87.706650,41.931902,-87.701195,member,6.82,0
1,8B38081EBE918800,electric_bike,2025-04-23 17:48:51.863,2025-04-23 17:59:06.015,Wabash Ave & Adams St,KA1503000015,Green St & Madison St,TA1307000120,41.879472,-87.625689,41.881859,-87.649264,member,10.24,3
2,1C7F1DE826BBBC8D,electric_bike,2025-04-05 17:55:30.845,2025-04-05 18:05:40.032,Damen Ave & Cortland St,13133,California Ave & Fletcher St,15642,41.915983,-87.677335,41.938429,-87.698008,member,10.15,6
3,CAD23D69A79A6C3B,classic_bike,2025-04-03 08:22:04.493,2025-04-03 08:32:06.099,Clark St & Elm St,TA1307000039,Orleans St & Merchandise Mart Plaza,TA1305000022,41.902973,-87.631280,41.888243,-87.636390,member,10.03,4
4,BE241E601482E0AB,electric_bike,2025-04-15 06:09:55.293,2025-04-15 06:19:58.942,Western Ave & Walton St,KA1504000103,Damen Ave & Charleston St,13288,41.898418,-87.686596,41.920082,-87.677855,member,10.06,2
5,752C0388D68EBBE5,electric_bike,2025-04-16 08:33:34.516,2025-04-16 08:41:49.998,Western Ave & Walton St,KA1504000103,Damen Ave & Charleston St,13288,41.898418,-87.686596,41.920082,-87.677855,member,8.26,3
6,C2C6CE2F046E4B4A,classic_bike,2025-04-26 17:11:59.253,2025-04-26 17:29:09.295,Lincoln Ave & Roscoe St*,chargingstx5,Broadway & Cornelia Ave,13278,41.943350,-87.670668,41.945529,-87.646439,member,17.17,6
7,D1004EF94E7957C3,electric_bike,2025-04-19 15:28:47.620,2025-04-19 15:49:30.801,Damen Ave & Cortland St,13133,Broadway & Wilson Ave,13074,41.915983,-87.677335,41.965221,-87.658139,member,20.72,6
8,95DC50DED42F7876,classic_bike,2025-04-29 19:19:32.788,2025-04-29 19:24:02.678,Clark St & Elm St,TA1307000039,Clark St & North Ave,13128,41.903322,-87.632999,41.911974,-87.631942,member,4.50,2
9,5CB06C086A4C8C21,classic_bike,2025-04-09 16:08:21.766,2025-04-09 16:52:59.793,Michigan Ave & Madison St,13036,Wabash Ave & Grand Ave,TA1307000117,41.882134,-87.625125,41.891466,-87.626761,casual,44.63,3


### 4.3 Remover registros com duração inválida

Com a coluna `ride_duration_min` criada, podemos verificar a distribuição das
durações e identificar valores que não representam viagens reais.  

Embora a Divvy, em sua página oficial, afirme que já removeu viagens com duração
igual ou inferior a 60 segundos — classificando-os como potenciais falsas partidas
ou tentativas de redocagem — esses registros continuam presentes no dataset.
Ou o filtro foi aplicado e esses dados por algum motivo passaram por ele,
ou o filtro não foi aplicado de forma consistente.
Estamos diante de vários 'ses'. Sem poder consultar diretamente a fonte,
optamos por eliminar apenas os registros factualmente impossíveis (duração ≤ 0)
e operacionalmente inválidos por limite regulatório do produto (> 1.440 minutos,
equivalente a 24 horas — prazo máximo de uso antes de cobranças por perda ou roubo).
Caso haja atualização na documentação ou no dataset, este tópico poderá ser revisitado.


Com esse disclaimer feito, vamos assumir que viagens com duração 0 ou maior
que 1.440 minutos (24 horas) serão os únicos registros descartados, caso existam.


### 4.4 Remoção de Durações Inválidas

Conforme decidido na etapa anterior, removemos os registros com `ride_duration_min <= 0`
(fisicamente impossíveis) e `ride_duration_min > 1440`
(acima de 24 horas — limite operacional do produto antes de cobranças por perda ou roubo).


In [28]:
#Removendo durações impossíveis
con.execute("""
    CREATE OR REPLACE TABLE etapa_03 AS
    SELECT *
    FROM etapa_02
    WHERE ride_duration_min > 0
      AND ride_duration_min <= 1440
""")

con.execute("""
    SELECT
        (SELECT COUNT(*) FROM etapa_02) AS registros_etapa_02,
        (SELECT COUNT(*) FROM etapa_03) AS registros_etapa_03,
        (SELECT COUNT(*) FROM etapa_02) - (SELECT COUNT(*) FROM etapa_03) AS removidos_etapa_03
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,registros_etapa_02,registros_etapa_03,removidos_etapa_03
0,5614647,5614399,248


Durações fora do intervalo válido (0 a 1440 minutos) foram removidas.
A diferença entre `registros_stage_02` e `registros_stage_03` indica o volume filtrado nesta etapa.


### 4.5 Padronização de Campos de Texto

Removemos espaços em branco extras (`TRIM`) dos campos de nome e ID de estação.  
Esse tipo de inconsistência não é visível diretamente, mas causa agrupamentos
incorretos na análise — `"Main St "` e `"Main St"` seriam tratados como estações distintas.


In [29]:
#Removendo espaço em branco na string
con.execute("""
    CREATE OR REPLACE TABLE etapa_04 AS
    SELECT
        ride_id,
        rideable_type,
        started_at,
        ended_at,
        CASE WHEN start_station_name IS NULL THEN NULL ELSE TRIM(start_station_name) END AS start_station_name,
        CASE WHEN start_station_id   IS NULL THEN NULL ELSE TRIM(start_station_id)   END AS start_station_id,
        CASE WHEN end_station_name   IS NULL THEN NULL ELSE TRIM(end_station_name)   END AS end_station_name,
        CASE WHEN end_station_id     IS NULL THEN NULL ELSE TRIM(end_station_id)     END AS end_station_id,
        start_lat,
        start_lng,
        end_lat,
        end_lng,
        member_casual,
        ride_duration_min,
        day_of_week
    FROM etapa_03
""")

#Checando o resultado
con.execute("""
    SELECT COUNT(*) AS registros_com_espacos
    FROM etapa_04
    WHERE (start_station_name IS NOT NULL AND start_station_name != TRIM(start_station_name))
       OR (end_station_name   IS NOT NULL AND end_station_name   != TRIM(end_station_name))
       OR (start_station_id   IS NOT NULL AND start_station_id   != TRIM(start_station_id))
       OR (end_station_id     IS NOT NULL AND end_station_id     != TRIM(end_station_id))
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,registros_com_espacos
0,0


A padronização foi aplicada. O valor `registros_com_espacos` confirma que
nenhum espaço extra permanece nas colunas de estação.


### Resultado Final

Os registros válidos após todas as etapas de limpeza são publicados na tabela
`cyclistic`, que reúne apenas as viagens com dados íntegros e com as colunas
derivadas necessárias para a análise.


In [30]:
con.execute("CREATE OR REPLACE TABLE cyclistic AS SELECT * FROM etapa_04")

con.execute("SELECT COUNT(*) AS registros_finais FROM cyclistic").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,registros_finais
0,5614399


In [31]:
#Checagens finais
con.execute("DESCRIBE SELECT * FROM cyclistic").df()

,column_name,column_type,null,key,default,extra
0,ride_id,VARCHAR,YES,None,None,None
1,rideable_type,VARCHAR,YES,None,None,None
2,started_at,TIMESTAMP,YES,None,None,None
3,ended_at,TIMESTAMP,YES,None,None,None
4,start_station_name,VARCHAR,YES,None,None,None
5,start_station_id,VARCHAR,YES,None,None,None
6,end_station_name,VARCHAR,YES,None,None,None
7,end_station_id,VARCHAR,YES,None,None,None
8,start_lat,DOUBLE,YES,None,None,None
9,start_lng,DOUBLE,YES,None,None,None


In [32]:
#Resumo
con.execute("""
    SELECT 'arquivos_csv' AS etapa, COUNT(*) AS registros FROM arquivos_csv
    UNION ALL
    SELECT 'etapa_01', COUNT(*) FROM etapa_01
    UNION ALL
    SELECT 'etapa_02', COUNT(*) FROM etapa_02
    UNION ALL
    SELECT 'etapa_03', COUNT(*) FROM etapa_03
    UNION ALL
    SELECT 'etapa_04', COUNT(*) FROM etapa_04
    UNION ALL
    SELECT 'cyclistic', COUNT(*) FROM cyclistic
""").df()

,etapa,registros
0,arquivos_csv,5620544
1,etapa_01,5614647
2,etapa_02,5614647
3,etapa_03,5614399
4,etapa_04,5614399
5,cyclistic,5614399


## 5. Resumo da Limpeza

A tabela acima mostra o volume de registros em cada etapa do processo.  
A diferença entre `arquivos_csv` (dado bruto) e `cyclistic` (dado final)
representa o total de registros removidos por ferir alguma dimensão de qualidade.

O dataset final contém apenas viagens válidas, com as colunas derivadas
`ride_duration_min` e `day_of_week` adicionadas para a análise.  
Cada decisão de remoção está documentada nas seções anteriores com sua respectiva justificativa.
